# Headline Inspection Notebook

Tools for exploring and characterising the WSJ headline corpus before
committing to a filtering strategy.

**Sections:**
1. Load corpus (standalone — no main pipeline required)
2. Random sample browser — eyeball random headlines by decade
3. Length and structure diagnostics
4. Keyword frequency analysis
5. FinBERT embedding similarity to financial anchor phrases
6. Category tagger — label a sample as relevant / irrelevant / uncertain
7. Summary statistics to inform filtering decisions

## 1. Load corpus

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings("ignore")

# ── Paths — update if needed ───────────────────────────────────────────────────
DATA_DIR = Path("./data")
NEWS_PARQUETS = [
    DATA_DIR / "wsj_headlines_1965_2014_zstd.parquet",
    DATA_DIR / "wsj_headlines_2015_2026_zstd.parquet",
]
NEWS_DATE_COL = "date"
NEWS_TEXT_COL = "headline"

# ── Load ──────────────────────────────────────────────────────────────────────
frames = []
for path in NEWS_PARQUETS:
    if not path.exists():
        print(f"WARNING: {path.name} not found — skipping.")
        continue
    df = pd.read_parquet(path)
    df = df.rename(columns={NEWS_DATE_COL: "date", NEWS_TEXT_COL: "text"})
    df = df[["date", "text"]].dropna()
    df["date"] = pd.to_datetime(df["date"])
    frames.append(df)
    print(f"Loaded {len(df):,} articles from {path.name}")

news_df = (pd.concat(frames, ignore_index=True)
             .drop_duplicates(subset=["date","text"])
             .sort_values("date")
             .reset_index(drop=True))

news_df["year"]       = news_df["date"].dt.year
news_df["decade"]     = (news_df["year"] // 10 * 10).astype(str) + "s"
news_df["word_count"] = news_df["text"].str.split().str.len()
news_df["char_count"] = news_df["text"].str.len()

print(f"\nCombined: {len(news_df):,} headlines")
print(f"Date range: {news_df['date'].min().date()} → {news_df['date'].max().date()}")
print(f"\nBy decade:")
print(news_df.groupby('decade').size().to_string())

## 2. Random sample browser

Change `DECADE` and `N_SAMPLE` to browse different periods.
Re-run the cell to get a fresh random sample.

In [ ]:
DECADE   = "all"   # "1970s", "1980s", "1990s", "2000s", "2010s", "2020s", or "all"
N_SAMPLE = 50      # number of headlines to display

pool = news_df if DECADE == "all" else news_df[news_df["decade"] == DECADE]
sample = pool.sample(n=min(N_SAMPLE, len(pool)), random_state=np.random.randint(0, 9999))

print(f"Random sample ({DECADE}, n={len(sample)}):")
print(f"{'─'*80}")
for _, row in sample.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')}  {row['text']}")
print(f"{'─'*80}")

## 3. Length and structure diagnostics

Very short headlines (< 4 words) are often wire alerts, corrections, or
boilerplate with no information content. Very long ones are sometimes
concatenated lede text rather than headlines.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Headline length distribution", fontweight="bold")

ax = axes[0]
ax.hist(news_df["word_count"].clip(0, 30), bins=30, color="#2c5f8a", alpha=0.85, edgecolor="white")
ax.set_xlabel("Word count"); ax.set_ylabel("Frequency")
ax.set_title("Word count distribution")
ax.axvline(4, color="red", ls="--", lw=1.5, label="4-word cutoff")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
# By decade
decades = sorted(news_df["decade"].unique())
medians = [news_df[news_df["decade"]==d]["word_count"].median() for d in decades]
ax.bar(decades, medians, color="#4a9e6b", alpha=0.85, edgecolor="white")
ax.set_xlabel("Decade"); ax.set_ylabel("Median word count")
ax.set_title("Median headline length by decade")
ax.grid(alpha=0.3, axis="y")

ax = axes[2]
# Volume by year
news_df.groupby("year").size().plot(ax=ax, color="#e07b39", lw=2)
ax.set_xlabel("Year"); ax.set_ylabel("Article count")
ax.set_title("Corpus volume by year")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Flagged by length
too_short = news_df[news_df["word_count"] < 4]
too_long  = news_df[news_df["word_count"] > 25]
print(f"Headlines < 4 words:  {len(too_short):,} ({len(too_short)/len(news_df):.1%})")
print(f"Headlines > 25 words: {len(too_long):,}  ({len(too_long)/len(news_df):.1%})")
print(f"\nSample of very short headlines:")
for _, row in too_short.sample(min(10, len(too_short))).iterrows():
    print(f"  '{row['text']}'")
print(f"\nSample of very long headlines:")
for _, row in too_long.sample(min(5, len(too_long))).iterrows():
    print(f"  '{row['text'][:120]}'...")

## 4. Keyword frequency analysis

Most frequent words in the corpus. Use this to identify both high-signal
financial terms and high-noise non-financial terms that should inform
keyword filtering.

In [ ]:
# Standard stopwords + common English words to exclude from frequency count
STOPWORDS = {
    "the","a","an","in","of","to","and","for","on","at","by","with","from",
    "is","are","was","were","be","been","has","have","had","will","would",
    "its","it","as","that","this","he","she","they","their","his","her",
    "up","out","over","new","no","not","but","or","after","into","about",
    "more","than","s","u","n","says","said","say","--","-","'s",
}

all_words = []
for text in news_df["text"].dropna():
    tokens = re.findall(r"[a-zA-Z]{3,}", text.lower())
    all_words.extend([t for t in tokens if t not in STOPWORDS])

word_freq = Counter(all_words)
top_words = word_freq.most_common(80)

print("Top 80 words in corpus:")
print(f"{'─'*50}")
for i, (word, count) in enumerate(top_words):
    pct = count / len(news_df) * 100
    bar = "█" * int(pct / top_words[0][1] * news_df.shape[0] / len(news_df) * 20)
    print(f"  {word:<20s} {count:>8,}  ({pct:4.1f}% of headlines)")
    if (i+1) % 20 == 0:
        print()

In [ ]:
# ── Bigram frequency (more informative than unigrams) ────────────────────────
from collections import Counter

bigrams = []
for text in news_df["text"].dropna():
    tokens = [t for t in re.findall(r"[a-zA-Z]{3,}", text.lower())
              if t not in STOPWORDS]
    bigrams.extend([f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens)-1)])

top_bigrams = Counter(bigrams).most_common(40)
print("Top 40 bigrams:")
for phrase, count in top_bigrams:
    pct = count / len(news_df) * 100
    print(f"  {phrase:<30s} {count:>7,}  ({pct:.2f}%)")

## 5. FinBERT embedding similarity to financial anchors

For a random sample of headlines, compute cosine similarity to a set of
"anchor" phrases representing financial relevance. Headlines with low
similarity to all anchors are likely off-topic.

**Requires the main pipeline to have run** (uses `model` and `tokenizer`).
Skip this cell if running standalone.

In [ ]:
# ── Check if encoder is available ────────────────────────────────────────────
try:
    _ = model, tokenizer
    ENCODER_AVAILABLE = True
except NameError:
    ENCODER_AVAILABLE = False
    print("Encoder not available — run the main pipeline notebook first, or skip this cell.")

if ENCODER_AVAILABLE:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    ANCHOR_PHRASES = [
        "corporate earnings quarterly results",
        "dividend payout shareholder return",
        "federal reserve interest rate monetary policy",
        "economic growth GDP inflation outlook",
        "stock market equity trading",
        "merger acquisition corporate deal",
        "employment jobs unemployment labor market",
        "oil energy commodity prices",
    ]

    def embed_texts(texts, batch_size=64):
        all_emb = []
        model.eval()
        for i in range(0, len(texts), batch_size):
            batch  = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=64)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                out = model(**inputs)
            cls = out.last_hidden_state[:, 0, :].cpu().numpy()
            all_emb.append(cls)
        return np.vstack(all_emb)

    def cosine_sim(A, B):
        A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
        B = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
        return A @ B.T

    print("Embedding anchor phrases...")
    anchor_emb = embed_texts(ANCHOR_PHRASES)

    # Sample 2000 headlines for speed
    SAMPLE_SIZE = 2000
    sample_idx  = np.random.choice(len(news_df), SAMPLE_SIZE, replace=False)
    sample_df   = news_df.iloc[sample_idx].copy().reset_index(drop=True)

    print(f"Embedding {SAMPLE_SIZE} sampled headlines...")
    sample_emb  = embed_texts(sample_df["text"].tolist())

    # Max cosine similarity to any anchor
    sim_matrix  = cosine_sim(sample_emb, anchor_emb)  # (sample, n_anchors)
    sample_df["max_anchor_sim"] = sim_matrix.max(axis=1)
    sample_df["best_anchor"]    = [ANCHOR_PHRASES[i] for i in sim_matrix.argmax(axis=1)]

    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("FinBERT cosine similarity to financial anchor phrases", fontweight="bold")

    ax = axes[0]
    ax.hist(sample_df["max_anchor_sim"], bins=40, color="#2c5f8a", alpha=0.85, edgecolor="white")
    ax.set_xlabel("Max cosine similarity to any anchor")
    ax.set_ylabel("Count")
    ax.set_title("Similarity distribution")
    for thresh in [0.7, 0.75, 0.80]:
        frac = (sample_df["max_anchor_sim"] < thresh).mean()
        ax.axvline(thresh, ls="--", lw=1.2, label=f"<{thresh}: {frac:.1%} dropped")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    sample_df.groupby("best_anchor")["max_anchor_sim"].mean().sort_values().plot(
        kind="barh", ax=ax, color="#4a9e6b", alpha=0.85)
    ax.set_xlabel("Mean max similarity")
    ax.set_title("Headlines per anchor (best match)")
    ax.grid(alpha=0.3, axis="x")

    plt.tight_layout()
    plt.show()

    # Show least financially relevant headlines
    print(f"\nBottom 20 headlines by financial relevance (max anchor sim):")
    print(f"{'─'*70}")
    for _, row in sample_df.nsmallest(20, "max_anchor_sim").iterrows():
        print(f"  {row['date'].strftime('%Y-%m-%d')}  sim={row['max_anchor_sim']:.3f}  {row['text']}")

## 6. Manual label sample

Print a batch of headlines for manual review. Mark each as:
- `r` = relevant (financial, informative)
- `i` = irrelevant (off-topic — sports, entertainment, etc.)
- `u` = uncertain / borderline

Run the cell, copy the output, annotate in a text file, then
paste labels back in the next cell to compute statistics.

In [ ]:
LABEL_N      = 100    # how many to label
LABEL_DECADE = "all"  # focus on a specific decade or "all"
LABEL_SEED   = 42     # change seed for a fresh batch

pool   = news_df if LABEL_DECADE == "all" else news_df[news_df["decade"] == LABEL_DECADE]
label_sample = pool.sample(LABEL_N, random_state=LABEL_SEED).reset_index(drop=True)
label_sample["id"] = range(len(label_sample))

print(f"Label the following {LABEL_N} headlines.")
print(f"Copy output, annotate each line with r/i/u, save to a text file.")
print(f"{'─'*80}")
for _, row in label_sample.iterrows():
    print(f"[{row['id']:>3}] {row['date'].strftime('%Y-%m-%d')}  {row['text']}")

In [ ]:
# ── Paste labels here after manual annotation ─────────────────────────────────
# Format: one label per line, same order as above. r=relevant, i=irrelevant, u=uncertain
# Example:
#   MANUAL_LABELS = """
#   r
#   i
#   r
#   ...
#   """

MANUAL_LABELS = """
# paste labels here, one per line
"""

labels = [l.strip() for l in MANUAL_LABELS.strip().splitlines()
          if l.strip() and not l.strip().startswith("#")]

if labels and len(labels) == len(label_sample):
    label_sample["label"] = labels
    counts = label_sample["label"].value_counts()
    print(f"Label counts:")
    print(f"  Relevant:    {counts.get('r', 0):>4}  ({counts.get('r', 0)/len(labels):.1%})")
    print(f"  Irrelevant:  {counts.get('i', 0):>4}  ({counts.get('i', 0)/len(labels):.1%})")
    print(f"  Uncertain:   {counts.get('u', 0):>4}  ({counts.get('u', 0)/len(labels):.1%})")

    print(f"\nSample irrelevant headlines:")
    irr = label_sample[label_sample["label"] == "i"]
    for _, row in irr.head(20).iterrows():
        print(f"  {row['date'].strftime('%Y-%m-%d')}  {row['text']}")
else:
    print(f"Add labels above — need {len(label_sample)} labels, found {len(labels)}.")

## 7. Filter candidate evaluation

Test different filtering strategies on the corpus and report what fraction
each would drop and what it catches. Use this to calibrate before editing
the main pipeline.

In [ ]:
# ── Define filter candidates ──────────────────────────────────────────────────
FINANCIAL_KEYWORDS = {
    # macro/markets
    "earnings","profit","loss","revenue","sales","dividend","yield","rate",
    "inflation","gdp","growth","recession","federal","reserve","fed","treasury",
    "bond","equity","stock","market","index","fund","portfolio","investment",
    "quarter","annual","forecast","outlook","guidance","deficit","surplus",
    # corporate
    "company","companies","corp","corporation","inc","ltd","ceo","merger",
    "acquisition","takeover","ipo","shares","shareholders","buyback",
    "bankruptcy","debt","credit","loan","bank","banking","financial",
    # commodities / energy
    "oil","gas","energy","commodity","gold","dollar","currency","exchange",
    # employment
    "jobs","employment","unemployment","workers","wages","labor","payroll",
}

NON_FINANCIAL_SIGNALS = {
    "sports","game","team","coach","player","championship","tournament",
    "film","movie","actor","actress","oscar","emmy","grammy","album",
    "weather","hurricane","tornado","earthquake","flood","storm",
    "recipe","food","restaurant","chef","diet",
    "fashion","style","dress","wear",
    "obituary","funeral","memorial","dies","died",
}

def filter_has_financial_kw(text):
    tokens = set(re.findall(r"[a-zA-Z]+", text.lower()))
    return bool(tokens & FINANCIAL_KEYWORDS)

def filter_no_nonfinancial_kw(text):
    tokens = set(re.findall(r"[a-zA-Z]+", text.lower()))
    return not bool(tokens & NON_FINANCIAL_SIGNALS)

def filter_min_words(text, n=4):
    return len(text.split()) >= n

def filter_max_words(text, n=30):
    return len(text.split()) <= n

# ── Evaluate each filter on full corpus ───────────────────────────────────────
filters = {
    "Min 4 words":           lambda t: filter_min_words(t, 4),
    "Max 30 words":          lambda t: filter_max_words(t, 30),
    "Has financial keyword": filter_has_financial_kw,
    "No non-financial kw":   filter_no_nonfinancial_kw,
    "4≤words≤30":            lambda t: filter_min_words(t,4) and filter_max_words(t,30),
    "Financial kw + length": lambda t: (filter_has_financial_kw(t)
                                         and filter_min_words(t,4)
                                         and filter_max_words(t,30)),
}

print(f"{'Filter':<30} {'Keep':>8} {'Drop':>8} {'Keep%':>8}")
print(f"{'─'*60}")
print(f"{'[No filter]':<30} {len(news_df):>8,} {'0':>8} {'100.0%':>8}")

for name, fn in filters.items():
    mask  = news_df["text"].apply(fn)
    keep  = mask.sum()
    drop  = (~mask).sum()
    print(f"{name:<30} {keep:>8,} {drop:>8,} {keep/len(news_df):>7.1%}")

print(f"\nSample headlines DROPPED by 'Financial kw + length' filter:")
print(f"{'─'*70}")
strict_mask = news_df["text"].apply(filters["Financial kw + length"])
dropped = news_df[~strict_mask].sample(min(25, (~strict_mask).sum()),
                                        random_state=42)
for _, row in dropped.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')}  {row['text']}")

## 8. Wave coverage impact

Check that your chosen filter doesn't leave any survey waves with zero
articles. Paste in your wave dates or run the main pipeline first.

In [ ]:
# Requires all_results from main pipeline — skip if running standalone
try:
    for series_name, res in all_results.items():
        waves = res["waves_df"]
        print(f"\n{series_name} — articles per wave after each filter:")
        print(f"{'Wave':<12} {'Raw':>6}", end="")
        for name in list(filters.keys())[-2:]:   # show last 2 filters
            print(f"  {name[:14]:>14}", end="")
        print()
        print("─" * 60)
        for _, wave in waves.iterrows():
            wd   = wave["wave_date"]
            mask = (
                (news_df["date"] >= wd - pd.Timedelta(days=30)) &
                (news_df["date"] <  wd)
            )
            wave_news = news_df[mask]
            raw_n = len(wave_news)
            print(f"{wd.strftime('%Y-%m'):>12} {raw_n:>6}", end="")
            for fn in list(filters.values())[-2:]:
                kept = wave_news["text"].apply(fn).sum()
                flag = " ⚠" if kept == 0 else ""
                print(f"  {kept:>14}{flag}", end="")
            print()
except NameError:
    print("all_results not defined — run main pipeline first, or skip this cell.")